# Filter the dataset based on event type and opening

In [1]:
import pandas as pd

df = pd.read_parquet('../data/data_2025_01.parquet')
df.tail()

,game_id,event,white_elo,black_elo,opening,winner,moves
9982296,9999996,Rated Blitz game,1203,1190,Philidor Defense,1,e2e4 e7e5 g1f3 d7d6 b1c3 b8c6 f1b5 c8d7 d2d3 a...
9982297,9999997,Rated Blitz game,1586,1593,"Sicilian Defense: Delayed Alapin Variation, wi...",1,e2e4 c7c5 g1f3 e7e6 c2c3 d7d5 e4d5 e6d5 d2d4 c...
9982298,9999998,Rated Blitz game,2094,2106,Ruy Lopez: Berlin Defense,2,e2e4 b8c6 g1f3 e7e5 f1b5 g8f6 b5c6 b7c6 f3e5 f...
9982299,9999999,Rated Blitz game,1449,1532,Scandinavian Defense: Main Line,1,e2e4 d7d5 e4d5 d8d5 b1c3 d5a5 d2d3 c7c6 c1d2 g...
9982300,10000000,Rated Blitz game,2069,2063,Sicilian Defense: Kramnik Variation,1,e2e4 c7c5 g1f3 e7e6 c2c4 b8c6 d2d3 d7d6 b1c3 g...


## Filter events

In [2]:
df["event"].value_counts()

event
Rated Blitz game                                                  4231136
Rated Bullet game                                                 3355083
Rated Rapid game                                                  1340401
Rated Classical game                                                56249
Rated UltraBullet game                                              46917
                                                                   ...   
Classical swiss https://lichess.org/swiss/bUvNze57                      1
Rated Rapid tournament https://lichess.org/tournament/wmyH1zNY          1
Rated Rapid tournament https://lichess.org/tournament/3XeES9oL          1
Blitz swiss https://lichess.org/swiss/UUugSTaP                          1
Rapid swiss https://lichess.org/swiss/WReDpHLY                          1
Name: count, Length: 4664, dtype: int64

In [3]:
events_to_keep = df["event"].value_counts().reset_index()["event"].head(4)
df_events = df[df["event"].isin(events_to_keep)]
df_events.head()

,game_id,event,white_elo,black_elo,opening,winner,moves
0,2,Rated Blitz game,1247,1218,Vienna Game: Anderssen Defense,1,b1c3 e7e5 e2e4 f8c5 d1h5 g8f6 h5e5 c5e7 d2d3 d...
1,3,Rated Blitz game,1577,1593,Caro-Kann Defense: Masi Variation,2,d2d4 c7c6 e2e4 g8f6 e4e5 f6g8 g1f3 d7d5 b1c3 c...
2,4,Rated Blitz game,1043,1000,Queen's Pawn Game,1,d2d4 d7d5 c2c3 b8c6 g1f3 c8g4 h2h3 g4f3 e2f3 e...
3,5,Rated Blitz game,2015,2028,Caro-Kann Defense: Exchange Variation,1,e2e4 c7c6 d2d4 d7d5 e4d5 c6d5 f1d3 g8f6 h2h3 b...
4,6,Rated Blitz game,2139,2145,Caro-Kann Defense: Endgame Variation,2,e2e4 c7c6 d2d3 d7d5 g1f3 d5e4 d3e4 d8d1 e1d1 g...


## Filter openings
To keep things simple, I’ve chosen a single, self-contained opening as the tree’s root - one with relatively few branching lines, unlike the Queen’s Pawn Game, which immediately spawns countless variations.

In [4]:
df_events["opening"].value_counts()

opening
Queen's Pawn Game                                           233844
Caro-Kann Defense                                           167472
Van't Kruijs Opening                                        149672
Philidor Defense                                            149371
Modern Defense                                              148902
                                                             ...  
Pterodactyl Defense: Central, Quetzalcoatlus Gambit              1
Latvian Gambit: Lobster Gambit                                   1
Dresden Opening: The Goblin                                      1
Italian Game: Classical Variation, Eisinger Variation            1
King's Gambit Declined: Classical Variation, Euwe Attack         1
Name: count, Length: 2900, dtype: int64

In [5]:
top10_openings = (
    df_events["opening"]
    .value_counts()
    .nlargest(10)
    .index
    .tolist()
)

df_top10 = df[df['opening'].isin(top10_openings)].copy()

def first_n_plies(moves: str, n: int) -> str:
    tokens = moves.split()
    return ' '.join(tokens[:n])

df_top10["first_5_moves"] = df_top10['moves'].apply(lambda mv: first_n_plies(mv, n=10))

variation_counts_top10 = (
    df_top10
    .groupby("opening")["first_5_moves"]
    .nunique()
    .reset_index(name="variation_count")
    .sort_values("variation_count")
)

min_var = variation_counts_top10["variation_count"].min()
best_openings_top10 = variation_counts_top10[variation_counts_top10["variation_count"] == min_var]

print("Top 10 Opening Lines - Variation Counts:")
print(variation_counts_top10.to_string(index=False))

print(f"\nOpening Line with the fewest variations ({min_var} variations):")
print(best_openings_top10.to_string(index=False))
sample_opening = best_openings_top10["opening"].iloc[0]

Top 10 Opening Lines - Variation Counts:
                                      opening  variation_count
Scandinavian Defense: Mieses-Kotroc Variation            33232
             French Defense: Knight Variation            48995
                             Philidor Defense            51761
 Queen's Pawn Game: Accelerated London System            58924
                            Caro-Kann Defense            62496
                         Scandinavian Defense            74946
                                 Pirc Defense            86145
                               Modern Defense            87653
                         Van't Kruijs Opening           156395
                            Queen's Pawn Game           183829

Opening Line with the fewest variations (33232 variations):
                                      opening  variation_count
Scandinavian Defense: Mieses-Kotroc Variation            33232


In [6]:
df_filtered = df_events[df_events["opening"] == sample_opening]
df_filtered.head()

,game_id,event,white_elo,black_elo,opening,winner,moves
46,48,Rated Bullet game,1921,1934,Scandinavian Defense: Mieses-Kotroc Variation,1,e2e4 d7d5 e4d5 d8d5 g1f3 d5f3 d1f3 c8g4 f3g4 g...
106,108,Rated Bullet game,1312,1371,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 c2c4 d5a5 b1c3 c7c6 d2d4 a...
108,110,Rated Bullet game,1928,1916,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 d2d4 c8f5 b1c3 d5a5 f2f4 c...
283,285,Rated Blitz game,1714,1705,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 d2d4 d5d6 b1c3 c8f5 f1c4 b...
402,404,Rated Blitz game,1333,1335,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 b1c3 d5e5 g1e2 c8g4 d2d4 e...


In [7]:
print(f"Dataset initially contained {len(df)} entries.")
print(f"After filtering, {len(df_filtered)} entries remain.")

Dataset initially contained 9982301 entries.
After filtering, 121082 entries remain.


## Calc average Elo


In [8]:
df_moves = df_filtered.copy()
df_moves['avg_elo'] = df_moves[['white_elo', 'black_elo']].mean(axis=1)
df_moves.head()

,game_id,event,white_elo,black_elo,opening,winner,moves,avg_elo
46,48,Rated Bullet game,1921,1934,Scandinavian Defense: Mieses-Kotroc Variation,1,e2e4 d7d5 e4d5 d8d5 g1f3 d5f3 d1f3 c8g4 f3g4 g...,1927.5
106,108,Rated Bullet game,1312,1371,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 c2c4 d5a5 b1c3 c7c6 d2d4 a...,1341.5
108,110,Rated Bullet game,1928,1916,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 d2d4 c8f5 b1c3 d5a5 f2f4 c...,1922.0
283,285,Rated Blitz game,1714,1705,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 d2d4 d5d6 b1c3 c8f5 f1c4 b...,1709.5
402,404,Rated Blitz game,1333,1335,Scandinavian Defense: Mieses-Kotroc Variation,2,e2e4 d7d5 e4d5 d8d5 b1c3 d5e5 g1e2 c8g4 d2d4 e...,1334.0


In [9]:
df_moves = df_moves[["avg_elo", "moves", "winner"]]
df_moves.head()

,avg_elo,moves,winner
46,1927.5,e2e4 d7d5 e4d5 d8d5 g1f3 d5f3 d1f3 c8g4 f3g4 g...,1
106,1341.5,e2e4 d7d5 e4d5 d8d5 c2c4 d5a5 b1c3 c7c6 d2d4 a...,2
108,1922.0,e2e4 d7d5 e4d5 d8d5 d2d4 c8f5 b1c3 d5a5 f2f4 c...,2
283,1709.5,e2e4 d7d5 e4d5 d8d5 d2d4 d5d6 b1c3 c8f5 f1c4 b...,2
402,1334.0,e2e4 d7d5 e4d5 d8d5 b1c3 d5e5 g1e2 c8g4 d2d4 e...,2


In [10]:
df_moves.to_parquet("../data/moves_2025_01.parquet")
print("✅ Saved moves data to '../data/moves_2025_01.parquet'")

✅ Saved moves data to '../data/moves_2025_01.parquet'
